# 04 · Retos del Workshop RAG

Retos opcionales (tope 30 pts). Cada reto es independiente salvo donde se indica
que reutiliza variables del notebook 03 (`embeddings`, `EMBEDDING_DIM`,
`format_docs`, `prompt`, `llm`, `attention_text`, `rag_text`).

- **C4** — Evaluación automática (`eval/preguntas.csv`).
- **C6** — Persistencia del índice en disco.
- **C5** — MMR vs similitud pura.
- **C1** — HNSW vs fuerza bruta con FAISS.
- **C3 (filtro)** — Snippet del filtro por metadata para la app de papers.

> Ejecuta primero las Partes 0–4 del notebook 03 en la **misma sesión** para
> tener las variables disponibles, o pega aquí las celdas de carga necesarias.

---
## C4 · Evaluación automática de la recuperación

Lee `eval/preguntas.csv` (10 preguntas con `fuente_esperada` y `palabra_clave`)
y calcula `hit@k`: porcentaje de preguntas cuyos chunks recuperados contienen la
palabra clave y vienen de la fuente esperada.

In [ ]:
import csv
from pathlib import Path

# Sube eval/preguntas.csv a la sesión de Colab (o ajusta la ruta).
CSV_PATH = Path("preguntas.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("eval/preguntas.csv")

with CSV_PATH.open(encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(f"{len(rows)} preguntas cargadas")


def hit_at_k(retriever, filas):
    hits = 0
    for r in filas:
        docs = retriever.invoke(r["pregunta"])
        ok = any(r["palabra_clave"].lower() in d.page_content.lower()
                 and d.metadata["source"] == r["fuente_esperada"] for d in docs)
        hits += ok
    return hits / len(filas)

In [ ]:
# Requiere build_store / build_chain del notebook 03 (celda B2).
print(f"{'chunk_size':>12} | {'hit@3':>6} | {'hit@5':>6}")
for cs, ov in [(400, 52), (900, 120), (1500, 195)]:
    store, _ = build_store(cs, ov)
    _, r3 = build_chain(store, k=3, threshold=0.0)
    _, r5 = build_chain(store, k=5, threshold=0.0)
    print(f"{cs:>12} | {hit_at_k(r3, rows):>6.2f} | {hit_at_k(r5, rows):>6.2f}")

# Dos limitaciones de esta métrica (para RESPUESTAS.md):
# 1) Sólo mira coincidencia literal de la palabra clave (no semántica).
# 2) Evalúa la recuperación, no la calidad de la respuesta final del LLM.

---
## C6 · Persistencia del índice en disco

Cambia `QdrantClient(":memory:")` por un cliente con ruta local. Indexa una vez;
al reabrir, comprueba si la colección existe y evita reindexar.

In [ ]:
import time
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

PERSIST_PATH = "./qdrant_data"
COLLECTION_P = "papers_persistente"

client_p = QdrantClient(path=PERSIST_PATH)

t0 = time.time()
if client_p.collection_exists(COLLECTION_P):
    store_p = QdrantVectorStore(client=client_p, collection_name=COLLECTION_P, embedding=embeddings)
    modo = "REABIERTO (sin reindexar)"
else:
    client_p.create_collection(collection_name=COLLECTION_P,
                               vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    store_p = QdrantVectorStore(client=client_p, collection_name=COLLECTION_P, embedding=embeddings)
    store_p.add_documents(all_docs)   # all_docs del notebook 03
    modo = "INDEXADO (primera vez)"
dt = time.time() - t0

print(f"{modo} en {dt:.2f} s")
print(store_p.similarity_search("What is the Transformer architecture?", k=2)[0].page_content[:200])

# Tabla de tiempos (para RESPUESTAS.md): compara "INDEXADO" vs "REABIERTO".
# Falta para producción: concurrencia multiusuario, control de acceso, respaldos,
# y en Colab el disco se borra al cerrar la sesión (montar Drive para persistir).

---
## C5 · MMR frente a similitud pura

MMR (Maximal Marginal Relevance) penaliza chunks redundantes. Compara los chunks
recuperados por similitud pura vs MMR para 3 preguntas.

In [ ]:
sim_retr = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
mmr_retr = vectorstore.as_retriever(
    search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5})

preguntas_c5 = [q1, q4, q_bonus]  # variadas: específica, de lista, cruzada

for q in preguntas_c5:
    print("\n" + "=" * 70)
    print("PREGUNTA:", q[:80])
    for nombre, retr in [("SIMILITUD", sim_retr), ("MMR", mmr_retr)]:
        print(f"\n  [{nombre}]")
        for d in retr.invoke(q):
            print(f"    - {d.metadata['source']}#{d.metadata['chunk_index']}: {d.page_content[:70]}...")

# Conclusión (RESPUESTAS.md): MMR ayuda cuando la pregunta abarca varios subtemas
# (mejor cobertura, menos duplicados); puede empeorar en preguntas muy puntuales
# donde el mejor chunk único basta y MMR mete ruido por diversificar.

---
## C1 · HNSW contra fuerza bruta con FAISS

Compara `IndexFlatL2` (exacto) con `IndexHNSWFlat` variando `efSearch` y `M`.
Mide recall@10, ms/consulta y tiempo de construcción. Luego grafica y barre `n`.

In [ ]:
%pip install -q faiss-cpu matplotlib

In [ ]:
import time
import faiss
import numpy as np

d, n, nq, k = 64, 20_000, 200, 10
rng = np.random.default_rng(42)
xb = rng.random((n, d), dtype="float32")
xq = rng.random((nq, d), dtype="float32")

flat = faiss.IndexFlatL2(d)
flat.add(xb)
t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000


def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))


rows = []
for M in (8, 16, 32):
    index = faiss.IndexHNSWFlat(d, M)
    index.hnsw.efConstruction = 100
    t0 = time.time(); index.add(xb); build_s = time.time() - t0
    for ef in (10, 16, 32, 64, 128, 256):
        index.hnsw.efSearch = ef
        t0 = time.time(); _, I = index.search(xq, k); ms = (time.time() - t0) / nq * 1000
        rows.append((M, ef, recall_at_k(I, I_true), ms, build_s))

print(f"exacto: {t_flat:.3f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@{k}={rec:.3f}  {ms:.3f} ms/q  (build {b:.1f}s)")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for M in (8, 16, 32):
    sub = [(ef, rec, ms) for (MM, ef, rec, ms, _) in rows if MM == M]
    efs = [x[0] for x in sub]; recs = [x[1] for x in sub]; mss = [x[2] for x in sub]
    ax1.plot(efs, recs, marker="o", label=f"M={M}")
    ax2.plot(mss, recs, marker="o", label=f"M={M}")
ax1.axhline(0.95, ls="--", c="gray"); ax1.set_xlabel("efSearch"); ax1.set_ylabel("recall@10"); ax1.set_title("recall vs efSearch"); ax1.legend()
ax2.set_xlabel("ms/consulta"); ax2.set_ylabel("recall@10"); ax2.set_title("recall vs latencia"); ax2.legend()
plt.tight_layout(); plt.show()

# Respuestas (RESPUESTAS.md):
# - ¿qué efSearch para recall>=0.95 con M=16? -> leer la curva M=16.
# - barrido de n (abajo) para ver desde qué n HNSW gana al exacto.
# - datos aleatorios = caso difícil (sin estructura/clusters); con embeddings de
#   texto reales, que sí tienen estructura semántica, HNSW alcanza alto recall
#   con menos esfuerzo.

In [ ]:
# Barrido de n: ¿desde qué tamaño HNSW es más rápido que el exacto?
for N in (20_000, 100_000, 500_000):
    xb2 = rng.random((N, d), dtype="float32")
    flat2 = faiss.IndexFlatL2(d); flat2.add(xb2)
    t0 = time.time(); flat2.search(xq, k); tf = (time.time() - t0) / nq * 1000
    h = faiss.IndexHNSWFlat(d, 16); h.hnsw.efConstruction = 100; h.add(xb2); h.hnsw.efSearch = 64
    t0 = time.time(); h.search(xq, k); th = (time.time() - t0) / nq * 1000
    print(f"n={N:>7,}  exacto={tf:.3f} ms/q   HNSW(M=16,ef=64)={th:.3f} ms/q   gana={'HNSW' if th < tf else 'exacto'}")

---
## C3 · Filtro por paper (snippet para la app Gradio)

Snippet para filtrar la recuperación por `metadata.source` usando un filtro de
Qdrant. La app Gradio completa se arma combinando este `retriever_for` con la
interfaz del notebook 02b.

In [ ]:
from qdrant_client import models

def retriever_for(source: str | None, k: int = 5, threshold: float = 0.25):
    kwargs = {"k": k, "score_threshold": threshold}
    if source and source != "Todos":
        kwargs["filter"] = models.Filter(must=[
            models.FieldCondition(key="metadata.source",
                                  match=models.MatchValue(value=source))
        ])
    return vectorstore.as_retriever(search_type="similarity_score_threshold",
                                    search_kwargs=kwargs)

# Prueba: misma pregunta con y sin filtro
for src in ("Todos", "attention_paper.pdf", "rag_paper.pdf"):
    r = retriever_for(src)
    docs = r.invoke("What architecture is used?")
    print(f"\nfiltro={src}: {len(docs)} chunks ->",
          [d.metadata["source"] for d in docs])

# Si el filtro no devuelve nada, inspecciona el payload guardado:
# print(qdrant_client.scroll(COLLECTION, limit=1))